In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
from statsmodels.stats.proportion import proportions_ztest

#数据导入部分

In [3]:
df = pd.read_csv('ab_data.csv')
countries = pd.read_csv('countries.csv')

In [4]:
df = df.merge(countries, on='user_id', how='left')
df

,user_id,timestamp,group,landing_page,converted,country
0,851104,11:48.6,control,old_page,0,US
1,804228,01:45.2,control,old_page,0,US
2,661590,55:06.2,treatment,new_page,0,US
3,853541,28:03.1,treatment,new_page,0,US
4,864975,52:26.2,control,old_page,1,US
...,...,...,...,...,...,...
294477,697314,20:29.0,control,old_page,0,US
294478,715931,40:24.5,treatment,new_page,0,UK
294479,759899,20:29.0,treatment,new_page,0,US
294480,759899,20:29.0,treatment,new_page,0,US


In [5]:
df.rename(columns={
    'group': 'group_name',
    'landing_page': 'landing_page',
    'converted': 'converted',
    'country': 'country'
}, inplace=True)

In [15]:
datagrip_project_path = r"C:\Users\戴欣言\DataGripProjects\mysql-base"
os.chdir(datagrip_project_path)

In [16]:
engine = create_engine('sqlite:///ab_test.db')

In [17]:
df.to_sql('ab_test', engine, if_exists='replace', index=False)

294482

#假设检验部分

In [6]:
#提取数据(Extract data)
control = df[df['group_name'] == 'control']
treatment = df[df['group_name'] == 'treatment']

In [7]:
#转化数与样本量 (Numbers of conversion and sample)
control_converted = control['converted'].sum()
control_n = len(control)
treatment_converted = treatment['converted'].sum()
treatment_n = len(treatment)

In [14]:
#Z test
diff = (treatment_c/treatment_n - control_c/control_n) * 100
counts = [control_converted, treatment_converted]
nobs = [control_n, treatment_n]
z_stat, p_value = proportions_ztest(counts, nobs)
print(f"差异={diff:.2f}%")
print(f"Z统计量: {z_stat:.4f}")
print(f"P值: {p_value:.4f}")

差异=0.08%
Z统计量: 0.6115
P值: 0.5409


H0: 实验组转化率 = 对照组转化率
H1: 实验组转化率 ！= 对照组转化率
Conclusion : p_value > 0.05
两组差异不显著，接受原假设。采用新页面的意义不大。

In [13]:
#按国家分层分析(Stratified analysis by country)
for country in df['country'].unique():
    df_c = df[df['country'] == country]
    control_c = df_c[df_c['group_name'] == 'control']['converted'].sum()
    control_n = len(df_c[df_c['group_name'] == 'control'])
    treatment_c = df_c[df_c['group_name'] == 'treatment']['converted'].sum()
    treatment_n = len(df_c[df_c['group_name'] == 'treatment'])
    z, p = proportions_ztest([control_c, treatment_c], [control_n, treatment_n])
    diff = (treatment_c/treatment_n - control_c/control_n) * 100
    print(f"{country}: 差异={diff:.2f}%,Z统计量={z:.4f}, P值={p:.4f}")

US: 差异=-0.19%,Z统计量=1.3357, P值=0.1816
CA: 差异=-0.67%,Z统计量=1.2769, P值=0.2016
UK: 差异=0.08%,Z统计量=-0.3254, P值=0.7449
